# PM Study Assistant — Agent Testing Notebook

This notebook tests the two agents (Router + Answerer) and the RAG pipeline
**directly**, outside Streamlit, so you can see each piece work on its own
and explain it confidently at your viva. It works by cloning your own
GitHub repo, so it always tests the exact same code that's in your
submission (no duplicated logic).

**Before running:** push your project to GitHub first (see GIT_COMMIT_PLAN.md),
and have your Groq + OpenRouter API keys ready.

In [ ]:
# 1. Clone your repo
# Replace <your-username> with your actual GitHub username
!git clone https://github.com/<your-username>/Personal-study-assistant-.git
%cd Personal-study-assistant-

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Set API keys for this Colab session only (never commit these!)
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your GROQ_API_KEY: ")
os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OPENROUTER_API_KEY: ")

## Step A — Test the RAG pipeline (chunking + embeddings + retrieval)

This builds the Chroma vector store from `corpus/*.md` the first time it
runs (takes ~30-60s to download the embedding model), then retrieves the
top-4 chunks for a sample query.

In [ ]:
from rag.retriever import retrieve

results = retrieve("What is scope creep?", k=4)
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['source']} | {r['heading']}]  distance={r['distance']:.3f}")
    print(r["text"][:200], "...\n")

## Step B — Test Agent 1: Router (Groq)

Confirms the router correctly classifies different question styles into
`concept_explanation`, `past_exam_question`, or `definition_lookup`, and
returns a structured `AgentMessage`.

In [ ]:
from agents.router_agent import route

test_queries = [
    "What is scope creep?",
    "Explain why IT projects fail more often than construction projects.",
    "Given PV=40000, EV=35000, AC=45000, BAC=100000, calculate CPI and EAC.",
]

for q in test_queries:
    msg = route(q)
    print(q)
    print(" ->", msg.to_dict())
    print()

## Step C — Test Agent 2: Answerer (RAG + OpenRouter + Reflection)

Takes the `AgentMessage` produced by the Router and generates a grounded
answer, showing the retrieved sources and the reflection/self-critique
result.

In [ ]:
from agents.router_agent import route
from agents.answerer_agent import answer

msg = route("What is the difference between contingency reserve and management reserve?")
result = answer(msg)

print("Category:", result["category"])
print()
print("Answer:")
print(result["answer"])
print()
print("Sources used:")
for s in result["sources"]:
    print(" -", s["source"], "|", s["heading"])
print()
print("Reflection:", result["reflection"])

## Step D — Full pipeline (Router -> Answerer via Orchestrator)

This is exactly what `app.py` calls when a user submits a question in
Streamlit.

In [ ]:
from orchestrator import run_pipeline

for q in [
    "What is scope creep?",
    "Which contract type suits a project with high technical uncertainty?",
]:
    print("Q:", q)
    result = run_pipeline(q)
    print("Detected category:", result["category"])
    print("Answer:", result["answer"][:400], "...")
    print("Router->Answerer message:", result["route_message"])
    print("=" * 80)

## What to look at before your viva

- Open `rag/ingest.py` and be ready to explain the chunking logic (section
  split + sliding window) line by line.
- Open `agents/protocol.py` and explain why a structured message (not just
  a plain string) is used between agents.
- Try changing something small right here in this notebook — e.g. change
  `k=4` to `k=6` in Step A, or add a 4th category to the router prompt in
  `agents/router_agent.py` — and re-run to see the effect. This is good
  practice for the "live modification" part of the viva.